# 04. Relevo e Topografia — SRTM 30m + Cruzamento com Focos, Clima e Cobertura

Análise da influência do relevo sobre os incêndios no Brasil, cruzando altitude e bandas topográficas com os dados dos notebooks anteriores.

**Fonte de altitude:** SRTM 30m via [OpenTopoData](https://www.opentopodata.org/) (API pública, sem cadastro)  
**Método:** amostragem estratificada por bioma (5.000 focos) → distribuição de altitude → cruzamento com FRP (nb01), ERA5 (nb02) e MapBiomas (nb03)  
**Período:** 2023 — mesmo dos notebooks 01–03

> **Nota metodológica:** A altitude é consultada para uma amostra estratificada proporcional por bioma (N=5 000). Estatísticas inferidas para o conjunto completo (189 901 focos) são obtidas via regressão simples altitude × bioma.

## 0. Dependências e configuração

In [ ]:
import time, urllib3
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from pathlib import Path

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

RELEVO_DIR = Path('data/relevo')
RELEVO_DIR.mkdir(parents=True, exist_ok=True)

ANO = 2023
N_AMOSTRA = 5_000  # pontos para consulta OpenTopoData

OTOPO_URL  = 'https://api.opentopodata.org/v1/srtm30m'
BATCH_SIZE = 100   # max por request
DELAY_S    = 1.1   # pausa entre requests (rate-limit)

MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

CORES_BIOMA = {
    'Amazonia':       '#2ca02c',
    'Cerrado':        '#ff7f0e',
    'Caatinga':       '#d62728',
    'Mata Atlantica': '#9467bd',
    'Pantanal':       '#17becf',
    'Pampa':          '#bcbd22',
}

# Normaliza nome do bioma para chave sem acento (usado internamente)
def bioma_key(b):
    return (b.replace('\u00e2','a').replace('\u00f4','o')
             .replace('\u00e3','a').replace('\u00e9','e')
             .replace('\u00e2','a').replace('\u00c2','A')
             .strip())

print('Configuracao OK')
print(f'RELEVO_DIR : {RELEVO_DIR}')
print(f'Periodo    : {ANO}')
print(f'N_AMOSTRA  : {N_AMOSTRA:,} pontos  (batches de {BATCH_SIZE})')

## 1. Altitude — amostragem SRTM 30m via OpenTopoData

In [ ]:
# Estatisticas de altitude por bioma — fallback hardcoded
# Fonte: IBGE / literatura (valores em metros)
# Usadas se OpenTopoData nao estiver disponivel
ALT_BIOMA_FALLBACK = {
    'Amazonia':       {'mediana': 90,   'q10': 20,  'q90': 280,  'std': 80},
    'Cerrado':        {'mediana': 780,  'q10': 400, 'q90': 1100, 'std': 220},
    'Caatinga':       {'mediana': 380,  'q10': 180, 'q90': 620,  'std': 140},
    'Mata Atlantica': {'mediana': 260,  'q10': 30,  'q90': 750,  'std': 260},
    'Pantanal':       {'mediana': 95,   'q10': 75,  'q90': 130,  'std': 18},
    'Pampa':          {'mediana': 140,  'q10': 50,  'q90': 300,  'std': 85},
}
print('Fallback de altitude por bioma:')
for b, v in ALT_BIOMA_FALLBACK.items():
    print(f'  {b:<20} mediana={v["mediana"]:>4}m  P10={v["q10"]:>4}m  P90={v["q90"]:>4}m')

In [ ]:
# Carrega focos do notebook 01 para criar a amostra estratificada
focos_csv = Path('data/inpe_queimadas/inpe_focos_2023_consolidado.csv')
df_focos_full = pd.read_csv(
    focos_csv,
    usecols=['latitude','longitude','bioma','frp','mes'],
    dtype={'frp': float, 'mes': int}
)
df_focos_full['bioma'] = df_focos_full['bioma'].str.strip()
print(f'Focos totais : {len(df_focos_full):,}')

# Amostra estratificada proporcional por bioma
def amostra_estratificada(df, n_total, col='bioma', seed=42):
    counts = df[col].value_counts()
    fracs  = counts / counts.sum()
    partes = []
    for bio, frac in fracs.items():
        n = max(1, round(frac * n_total))
        partes.append(df[df[col] == bio].sample(min(n, len(df[df[col]==bio])), random_state=seed))
    return pd.concat(partes).reset_index(drop=True)

df_sample = amostra_estratificada(df_focos_full, N_AMOSTRA)
print(f'Amostra      : {len(df_sample):,} pontos')
print('Distribuicao da amostra:')
print(df_sample['bioma'].value_counts().to_string())

In [ ]:
# Consulta altitude via OpenTopoData (SRTM 30m) em batches de 100 pontos
cache_alt = RELEVO_DIR / f'srtm_amostra_{ANO}.csv'

if cache_alt.exists():
    df_sample = pd.read_csv(cache_alt)
    print(f'[cache] Altitude carregada de {cache_alt}  ({len(df_sample):,} pontos)')
else:
    print(f'Consultando OpenTopoData para {len(df_sample):,} pontos...')
    altitudes = []
    batches = [df_sample.iloc[i:i+BATCH_SIZE] for i in range(0, len(df_sample), BATCH_SIZE)]
    erros = 0
    t0 = time.time()

    for idx, batch in enumerate(batches):
        locs = '|'.join(f'{r.latitude:.5f},{r.longitude:.5f}' for _, r in batch.iterrows())
        try:
            resp = requests.get(f'{OTOPO_URL}?locations={locs}', timeout=20)
            if resp.ok:
                results = resp.json().get('results', [])
                altitudes.extend(r['elevation'] for r in results)
            else:
                altitudes.extend([None] * len(batch))
                erros += 1
        except Exception as e:
            altitudes.extend([None] * len(batch))
            erros += 1

        if (idx + 1) % 10 == 0:
            pct = (idx+1) / len(batches) * 100
            print(f'  {pct:.0f}%  ({idx+1}/{len(batches)} batches)  erros={erros}')
        if idx < len(batches) - 1:
            time.sleep(DELAY_S)

    df_sample = df_sample.copy()
    df_sample['altitude_m'] = altitudes
    df_sample['altitude_m'] = pd.to_numeric(df_sample['altitude_m'], errors='coerce')
    df_sample.to_csv(cache_alt, index=False)

    n_ok  = df_sample['altitude_m'].notna().sum()
    print(f'\n[ok] {n_ok:,}/{len(df_sample):,} altitudes obtidas  ({100*n_ok/len(df_sample):.1f}%)')
    print(f'     Tempo total: {time.time()-t0:.0f}s  Erros: {erros}')

print(f'\nEstatisticas de altitude (amostra):')
print(df_sample['altitude_m'].describe().round(0).to_string())

## 2. Cruzamento com Notebooks 01, 02 e 03

In [ ]:
# 2a. Notebook 01 — focos INPE
# df_sample ja tem bioma, frp, mes (carregado acima)
print('=== Notebook 01 — Focos INPE ===')
print(f'Amostra com altitude: {len(df_sample):,} focos')
print(f'Biomas: {sorted(df_sample["bioma"].unique())}')
print(f'FRP: min={df_sample["frp"].min():.1f}  max={df_sample["frp"].max():.1f}  med={df_sample["frp"].median():.1f}')
df_sample[['latitude','longitude','bioma','frp','mes','altitude_m']].head()

In [ ]:
# 2b. Notebook 02 — ERA5 (temperatura, umidade, vento)
# ERA5: serie temporal diaria para o Brasil — agrega por mes
era5_csv = Path('data/era5/era5_2023_brasil_diario.csv')

df_era5 = None
if era5_csv.exists():
    df_era5 = pd.read_csv(era5_csv)
    print(f'ERA5 carregado: {len(df_era5):,} linhas  colunas={df_era5.columns.tolist()}')

    # Identifica coluna de data/mes
    col_data = next((c for c in df_era5.columns if 'dat' in c.lower() or 'time' in c.lower()), None)
    if col_data:
        df_era5[col_data] = pd.to_datetime(df_era5[col_data], errors='coerce')
        df_era5['mes'] = df_era5[col_data].dt.month

    # Media mensal das variaveis climaticas
    vars_clima = [c for c in ['temp_c','rh','wind_speed','ffwi','tp'] if c in df_era5.columns]
    if vars_clima and 'mes' in df_era5.columns:
        df_era5_mensal = df_era5.groupby('mes')[vars_clima].mean()
        print(f'ERA5 mensal ({vars_clima}):')
        print(df_era5_mensal.round(2).to_string())
    else:
        df_era5_mensal = None
        print('[aviso] Colunas climaticas nao encontradas no ERA5')
else:
    df_era5_mensal = None
    print(f'[aviso] {era5_csv} nao encontrado — execute o notebook 02 primeiro')
    print('         Cruzamento ERA5 x altitude sera omitido')

In [ ]:
# 2c. Notebook 03 — MapBiomas (cobertura do solo)
mapb_csv = Path('data/mapbiomas/focos_mapbiomas_2023.csv')

df_mapb = None
if mapb_csv.exists():
    df_mapb = pd.read_csv(mapb_csv, usecols=['latitude','longitude','bioma','classe_mb','grupo_mb'])
    print(f'MapBiomas carregado: {len(df_mapb):,} focos')
    print(f'Grupos: {sorted(df_mapb["grupo_mb"].unique())}')

    # Merge altitude com MapBiomas via lat/lon da amostra
    # (usa merge por lat/lon exato — a amostra e subconjunto de focos_mapbiomas)
    df_sample_mb = df_sample.merge(
        df_mapb[['latitude','longitude','classe_mb','grupo_mb']],
        on=['latitude','longitude'], how='left'
    )
    n_matched = df_sample_mb['grupo_mb'].notna().sum()
    print(f'Merge altitude x MapBiomas: {n_matched:,}/{len(df_sample):,} pontos cruzados')
else:
    df_sample_mb = df_sample.copy()
    df_sample_mb['classe_mb'] = None
    df_sample_mb['grupo_mb']  = None
    print(f'[aviso] {mapb_csv} nao encontrado — execute o notebook 03 primeiro')

## 3. Análise por bandas de altitude

In [ ]:
# Define bandas de altitude e classifica cada foco
BANDAS = [
    (  -999,    50, 'Planicie (<50m)'),
    (    50,   200, 'Baixada (50-200m)'),
    (   200,   500, 'Planalto Baixo (200-500m)'),
    (   500,   900, 'Planalto Alto (500-900m)'),
    (   900, 10000, 'Serra (>900m)'),
]

def classifica_banda(alt):
    if pd.isna(alt): return 'Sem dado'
    for lo, hi, nome in BANDAS:
        if lo < alt <= hi:
            return nome
    return 'Sem dado'

df_sample_mb['banda_alt'] = df_sample_mb['altitude_m'].apply(classifica_banda)

ordem_bandas = [b[2] for b in BANDAS]

print('Distribuicao de focos por banda de altitude (amostra):')
cnt_banda = df_sample_mb['banda_alt'].value_counts().reindex(ordem_bandas, fill_value=0)
for banda, n in cnt_banda.items():
    pct = 100 * n / len(df_sample_mb)
    print(f'  {banda:<30} {n:>5} focos  ({pct:.1f}%)')

print()
print('Altitude mediana por bioma (amostra):')
alt_bioma = (
    df_sample_mb.groupby('bioma')['altitude_m']
    .agg(['median','mean','std','min','max','count'])
    .rename(columns={'median':'mediana','mean':'media','std':'dp','min':'minimo','max':'maximo','count':'n'})
    .sort_values('mediana', ascending=False)
)
print(alt_bioma.round(0).to_string())

In [ ]:
# Correlacoes topografia x incendio
print('=== Correlacoes: altitude x variaveis de incendio ===')

# 1. Altitude x FRP
df_v = df_sample_mb[['altitude_m','frp']].dropna()
if len(df_v) >= 10:
    r_frp = df_v.corr().iloc[0,1]
    print(f'Altitude x FRP   : r = {r_frp:.3f}  (n={len(df_v):,})')

# 2. Altitude x mes (sazonalidade)
df_v2 = df_sample_mb[['altitude_m','mes']].dropna()
if len(df_v2) >= 10:
    r_mes = df_v2.corr().iloc[0,1]
    print(f'Altitude x Mes   : r = {r_mes:.3f}  (n={len(df_v2):,})')

# 3. FRP por banda de altitude
print()
print('FRP mediano por banda de altitude:')
frp_banda = (
    df_sample_mb.groupby('banda_alt')['frp']
    .agg(['median','count'])
    .reindex(ordem_bandas)
    .rename(columns={'median':'frp_mediano','count':'n_focos'})
)
print(frp_banda.round(1).to_string())

# 4. Sazonalidade por banda — mes de pico
print()
print('Mes de pico de focos por banda de altitude:')
pico_mes = (
    df_sample_mb.groupby(['banda_alt','mes'])
    .size()
    .reset_index(name='n')
    .sort_values('n', ascending=False)
    .groupby('banda_alt')
    .first()
    .reindex(ordem_bandas)
)
pico_mes['mes_nome'] = pico_mes['mes'].apply(lambda m: MESES_PT[int(m)-1] if pd.notna(m) else '-')
print(pico_mes[['mes_nome','n']].to_string())

In [ ]:
# Correlacao altitude x clima (ERA5) via bioma
# Usa altitude mediana por bioma (da amostra) e media anual de cada variavel

df_cross_clima = None
if df_era5_mensal is not None and len(alt_bioma) > 0:
    vars_clima = [c for c in ['temp_c','rh','wind_speed','ffwi'] if c in df_era5_mensal.columns]
    if vars_clima:
        era5_anual = df_era5_mensal[vars_clima].mean().to_dict()
        # Associa altitude mediana de cada bioma com media ERA5 do Brasil
        # (ERA5 nao esta dividido por bioma — usamos altitude como variavel explicativa)
        alt_bioma_r = alt_bioma.reset_index()[['bioma','mediana']].rename(columns={'mediana':'alt_mediana'})
        for v, val in era5_anual.items():
            alt_bioma_r[v] = val  # mesma valor ERA5 para todos (serie nacional)
        df_cross_clima = alt_bioma_r
        print('Altitude mediana por bioma + ERA5 anual:')
        print(df_cross_clima.to_string(index=False))
    else:
        print('[aviso] Variaveis climaticas nao encontradas no ERA5')
else:
    print('[info] ERA5 nao disponivel — cruzamento altitude x clima omitido')

## 4. Visualizações

In [ ]:
CORES_BANDA = {
    'Planicie (<50m)':            '#1f77b4',
    'Baixada (50-200m)':          '#aec7e8',
    'Planalto Baixo (200-500m)':  '#ff7f0e',
    'Planalto Alto (500-900m)':   '#d62728',
    'Serra (>900m)':              '#9467bd',
}

# Mapa bioma -> cor (normaliza nome)
def cor_bioma(nome):
    for k, v in CORES_BIOMA.items():
        if k.lower() in nome.lower() or nome.lower() in k.lower():
            return v
    return '#aaaaaa'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f'Topografia e Incendios — SRTM 30m x INPE/MapBiomas {ANO}',
    fontsize=13, fontweight='bold'
)

# 1. Distribuicao de altitude por bioma (boxplot)
ax = axes[0, 0]
biomas_ord = alt_bioma.sort_values('mediana', ascending=True).index.tolist()
dados_box = [df_sample_mb[df_sample_mb['bioma']==b]['altitude_m'].dropna().values for b in biomas_ord]
bp = ax.boxplot(dados_box, vert=True, patch_artist=True,
                medianprops={'color':'black','linewidth':1.5},
                whiskerprops={'linewidth':0.8}, capprops={'linewidth':0.8},
                flierprops={'marker':'.','markersize':2})
for flier in bp['fliers']:
    flier.set_alpha(0.3)
for patch, bioma in zip(bp['boxes'], biomas_ord):
    patch.set_facecolor(cor_bioma(bioma))
    patch.set_alpha(0.8)
ax.set_xticks(range(1, len(biomas_ord)+1))
rotulos = [b.replace(' ', '\n') for b in biomas_ord]
ax.set_xticklabels(rotulos, fontsize=7.5)
ax.set_ylabel('Altitude (m)')
ax.set_title('Distribuicao de Altitude por Bioma')
ax.grid(axis='y', alpha=0.25)

# 2. Focos por banda de altitude (por bioma, stacked)
ax = axes[0, 1]
pivot_banda = (
    df_sample_mb.groupby(['bioma','banda_alt'])
    .size().unstack(fill_value=0)
    .reindex(columns=ordem_bandas, fill_value=0)
)
pivot_banda_pct = pivot_banda.div(pivot_banda.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(pivot_banda_pct))
for banda in ordem_bandas:
    if banda in pivot_banda_pct.columns:
        vals = pivot_banda_pct[banda].values
        ax.bar(pivot_banda_pct.index, vals, bottom=bottom,
               color=CORES_BANDA.get(banda,'#aaaaaa'), label=banda,
               edgecolor='white', linewidth=0.4)
        bottom += vals
patches = [mpatches.Patch(color=CORES_BANDA.get(b,'#aaa'), label=b) for b in ordem_bandas]
ax.legend(handles=patches, fontsize=6.5, loc='lower right', framealpha=0.85)
ax.set_xticks(range(len(pivot_banda_pct)))
ax.set_xticklabels(pivot_banda_pct.index, rotation=18, ha='right', fontsize=8)
ax.set_ylabel('%')
ax.set_ylim(0, 100)
ax.set_title(f'Composicao de Altitude por Bioma (%) — n={len(df_sample_mb):,}')
ax.grid(axis='y', alpha=0.25)

# 3. FRP mediano por banda de altitude
ax = axes[1, 0]
frp_ok = frp_banda.dropna(subset=['frp_mediano'])
cores_banda_lista = [CORES_BANDA.get(b,'#aaaaaa') for b in frp_ok.index]
ax.barh(frp_ok.index, frp_ok['frp_mediano'], color=cores_banda_lista, edgecolor='white')
ax.set_xlabel('FRP mediano (MW)')
ax.set_title('FRP Mediano por Banda de Altitude')
for i, (_, row) in enumerate(frp_ok.iterrows()):
    ax.text(row['frp_mediano'] + 0.3, i, f'{row["frp_mediano"]:.1f}  (n={int(row["n_focos"]):,})',
            va='center', fontsize=7)
ax.grid(axis='x', alpha=0.25)

# 4. Sazonalidade por banda — focos por mes (linhas)
ax = axes[1, 1]
for banda, cor in CORES_BANDA.items():
    sub = df_sample_mb[df_sample_mb['banda_alt'] == banda]
    if len(sub) < 5: continue
    cnt_mes = sub.groupby('mes').size().reindex(range(1,13), fill_value=0)
    ax.plot(cnt_mes.index, cnt_mes.values, marker='o', markersize=4,
            color=cor, linewidth=1.5, label=banda.split('(')[0].strip())
ax.set_xticks(range(1,13))
ax.set_xticklabels(MESES_PT, fontsize=8)
ax.set_ylabel('N de focos (amostra)')
ax.set_title('Sazonalidade por Banda de Altitude')
ax.legend(fontsize=7, loc='upper left', framealpha=0.85)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(RELEVO_DIR / f'fig_painel_relevo_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Mapa espacial — focos coloridos por banda de altitude
fig, ax = plt.subplots(figsize=(10, 9))

for banda in ordem_bandas:
    sub = df_sample_mb[df_sample_mb['banda_alt'] == banda]
    if len(sub) == 0: continue
    ax.scatter(
        sub['longitude'], sub['latitude'],
        s=2, alpha=0.5, color=CORES_BANDA.get(banda,'#aaaaaa'),
        label=banda, rasterized=True
    )

ax.set_xlim(-74, -28); ax.set_ylim(-34, 6)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(
    f'Focos de Calor por Banda de Altitude — {ANO}  '
    f'(SRTM 30m | amostra n={len(df_sample_mb):,})',
    fontsize=11
)
ax.legend(markerscale=6, fontsize=8, loc='lower left', framealpha=0.85)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(RELEVO_DIR / f'fig_mapa_relevo_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico altitude x FRP (dispersao) e altitude x bioma (violino simplificado)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f'Relacao Relevo x Intensidade de Incendio — {ANO}  (SRTM 30m)',
    fontsize=12, fontweight='bold'
)

# 1. Scatter: altitude x FRP (por bioma)
ax = axes[0]
for bio in df_sample_mb['bioma'].unique():
    sub = df_sample_mb[df_sample_mb['bioma']==bio][['altitude_m','frp']].dropna()
    if len(sub) < 3: continue
    ax.scatter(sub['altitude_m'], sub['frp'], s=4, alpha=0.35,
               color=cor_bioma(bio), label=bio, rasterized=True)

# Linha de tendencia (regressao linear)
df_lm = df_sample_mb[['altitude_m','frp']].dropna()
if len(df_lm) >= 10:
    coef = np.polyfit(df_lm['altitude_m'], df_lm['frp'], 1)
    x_fit = np.linspace(df_lm['altitude_m'].min(), df_lm['altitude_m'].max(), 100)
    ax.plot(x_fit, np.polyval(coef, x_fit), 'k--', linewidth=1.2, label=f'Tendencia (b={coef[0]:.3f})')
    r = df_lm.corr().iloc[0,1]
    ax.set_title(f'Altitude x FRP  (r = {r:.2f})')

ax.set_xlabel('Altitude (m)')
ax.set_ylabel('FRP (MW)')
ax.legend(fontsize=7, loc='upper right', framealpha=0.85, markerscale=4)
ax.grid(True, alpha=0.25)

# 2. Altitude mediana por bioma (barras horizontais)
ax = axes[1]
alt_ord = alt_bioma.sort_values('mediana')
cores_ord = [cor_bioma(b) for b in alt_ord.index]
bars = ax.barh(alt_ord.index, alt_ord['mediana'], color=cores_ord, edgecolor='white')
# Erro (std)
ax.errorbar(alt_ord['mediana'], range(len(alt_ord)),
            xerr=alt_ord['dp'], fmt='none', color='gray', capsize=4, linewidth=1)
for i, (_, row) in enumerate(alt_ord.iterrows()):
    ax.text(row['mediana'] + 10, i, f'{row["mediana"]:.0f}m  (n={int(row["n"]):,})',
            va='center', fontsize=7.5)
ax.set_xlabel('Altitude mediana (m)')
ax.set_title('Altitude Mediana por Bioma (SRTM 30m)\n(barra de erro = desvio padrao)')
ax.grid(axis='x', alpha=0.25)

plt.tight_layout()
plt.savefig(RELEVO_DIR / f'fig_altitude_frp_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Estatísticas e exportação

In [ ]:
print(f'=== Estatisticas Relevo x Incendios — {ANO} ===')
print()

print('-- Altitude por bioma (m) --')
print(alt_bioma[['mediana','dp','minimo','maximo','n']].round(0).to_string())

print()
print('-- Focos por banda de altitude --')
tab_banda = (
    df_sample_mb.groupby('banda_alt')
    .agg(
        focos=('frp','count'),
        frp_mediano=('frp','median'),
        alt_mediana=('altitude_m','median'),
    )
    .reindex(ordem_bandas)
)
tab_banda['pct'] = (tab_banda['focos'] / tab_banda['focos'].sum() * 100).round(1)
print(tab_banda.round(1).to_string())

if 'grupo_mb' in df_sample_mb.columns and df_sample_mb['grupo_mb'].notna().any():
    print()
    print('-- Cobertura MapBiomas dominante por banda de altitude --')
    domina = (
        df_sample_mb.dropna(subset=['grupo_mb'])
        .groupby('banda_alt')['grupo_mb']
        .apply(lambda x: x.value_counts().index[0] if len(x) > 0 else '-')
        .reindex(ordem_bandas)
    )
    print(domina.to_string())


In [ ]:
# Exporta datasets
out_amostra  = RELEVO_DIR / f'focos_altitude_{ANO}.csv'
out_bioma    = RELEVO_DIR / f'altitude_bioma_{ANO}.csv'
out_banda    = RELEVO_DIR / f'altitude_banda_{ANO}.csv'

cols_export = [c for c in ['latitude','longitude','bioma','frp','mes','altitude_m','banda_alt','grupo_mb'] if c in df_sample_mb.columns]
df_sample_mb[cols_export].to_csv(out_amostra, index=False)
alt_bioma.reset_index().to_csv(out_bioma, index=False)
tab_banda.reset_index().to_csv(out_banda, index=False)

print('Arquivos exportados:')
for p in [out_amostra, out_bioma, out_banda]:
    print(f'  {p}  ({p.stat().st_size/1024:.0f} KB)')

print()
print('Figuras geradas:')
for fig_path in sorted(RELEVO_DIR.glob('fig_*.png')):
    print(f'  {fig_path}')

In [ ]:
try:
    import srtm
    SRTM_PKG_OK = True
except ImportError:
    SRTM_PKG_OK = False
    print('[aviso] instale: pip install srtm.py')

out_completo = RELEVO_DIR / f'focos_altitude_completo_{ANO}.csv'

if out_completo.exists():
    df_altitude_completo = pd.read_csv(out_completo)
    print(f'[cache] {out_completo.name}  ({len(df_altitude_completo):,} focos)')

elif SRTM_PKG_OK:
    print(f'Consultando SRTM para {len(df_focos_full):,} focos...')
    print('(primeira execucao baixa ~100 MB de tiles SRTM; progress a cada 10k focos)')
    tile_dir = RELEVO_DIR / 'srtm_tiles'
    tile_dir.mkdir(exist_ok=True)
    srtm_data = srtm.get_data(local_cache_dir=str(tile_dir))

    altitudes = []
    for i, (lat, lon) in enumerate(zip(df_focos_full['latitude'], df_focos_full['longitude'])):
        alt = srtm_data.get_elevation(lat, lon)
        altitudes.append(float(alt) if alt is not None else np.nan)
        if (i + 1) % 10_000 == 0:
            pct = (i + 1) / len(df_focos_full) * 100
            print(f'  {i+1:,}/{len(df_focos_full):,}  ({pct:.0f}%)')

    df_altitude_completo = df_focos_full[['latitude', 'longitude', 'bioma', 'frp', 'mes']].copy()
    df_altitude_completo['altitude_m'] = altitudes
    df_altitude_completo['altitude_m'] = pd.to_numeric(df_altitude_completo['altitude_m'], errors='coerce')

    n_ok = df_altitude_completo['altitude_m'].notna().sum()
    print(f'\n[ok] {n_ok:,}/{len(df_altitude_completo):,} altitudes ({100*n_ok/len(df_altitude_completo):.1f}%)')
    df_altitude_completo.to_csv(out_completo, index=False)
    print(f'Salvo: {out_completo}  ({out_completo.stat().st_size/1e6:.1f} MB)')

else:
    df_altitude_completo = None
    print('[aviso] srtm.py nao disponivel — instale e reexecute')

## 6. Enriquecimento completo — altitude para todos os focos

A análise topográfica anterior usou uma amostra de 5.000 focos. Para o dataset de consolidação (notebook 07), todos os 189.901 focos precisam de altitude.

**Método:** biblioteca `srtm.py` — baixa tiles SRTM1 (30 m) do USGS sob demanda e armazena em cache local (`data/relevo/srtm_tiles/`). Primeira execução baixa ~100 MB de tiles para o Brasil; subsequentes usam o cache e completam em segundos.